In [70]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init
from torch.autograd import Variable
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader
import torchvision
import torchvision.transforms as transforms
import pickle
import numpy as np

In [71]:
#torchvision.datasets.CIFAR10(root='./data', download=True) #run once

In [72]:
class Config:
    def __init__(self):
        # Data / paths
        self.input_channels = 3
        self.path_train = "data/cifar-10-batches-py/data_batch_1"
        self.path_test = "data/cifar-10-batches-py/test_batch"

        # Training
        self.epochs = 50
        self.batch_size = 128
        self.lr = 1e-3

        # Architecture
        self.relu_mode = 1   # 0,1,2
        self.blocks = [3, 3, 3]
        self.num_classes = 10

        # Dataset flags
        self.use_mnist = False
        self.use_cifar10 = True

        # Load Pretrained
        self.load_pretrained = True
        self.pretrained_path = "models/baseline_Baseline.pth"


config = Config()


In [73]:
def save_model(model, name,relu):
    if relu:
        torch.save(model.state_dict(), f'models/{name}_ReLU.pth')
    else:
        torch.save(model.state_dict(), f'models/{name}_noReLU.pth')

In [74]:
def load_model(path,model_type):
    model = model_type()
    model.load_state_dict(torch.load(f"models/{path}"))
    model.eval()
    return model

In [75]:
def load_batch(path):
    all_images = []
    all_labels = []
    for i in range(1, 6):
        path = f"data/cifar-10-batches-py/data_batch_{i}"
        with open(path, 'rb') as f:
            batch = pickle.load(f, encoding='bytes')
            
        images = batch[b'data']      # (10000, 3072)
        labels = batch[b'labels']    # length 10000

        images = images.reshape(10000, 3, 32, 32)
        labels = np.array(labels)
        all_images.append(images)
        all_labels.append(labels)
    
    X = np.concatenate(all_images)
    y = np.concatenate(all_labels)
    return X, y


In [76]:
class LambdaLayer(nn.Module):
    def __init__(self, lambd):
        super(LambdaLayer, self).__init__()
        self.lambd = lambd

    def forward(self, x):
        return self.lambd(x)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, config, stride=1, option='A'):
        super().__init__()
        self.config = config

        self.conv1r = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=True)
        init.kaiming_normal_(self.conv1r.weight)

        self.conv1g = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=True)
        self.conv1g.weight.data.fill_(0)
        self.conv1g.bias.data.fill_(1)

        self.conv1b = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=True)
        self.conv1b.weight.data.fill_(0)
        self.conv1b.bias.data.fill_(0)

        self.bn1 = nn.BatchNorm2d(planes)

        self.conv2r = nn.Conv2d(planes, planes, 3, 1, 1, bias=True)
        init.kaiming_normal_(self.conv2r.weight)

        self.conv2g = nn.Conv2d(planes, planes, 3, 1, 1, bias=True)
        self.conv2g.weight.data.fill_(0)
        self.conv2g.bias.data.fill_(1)

        self.conv2b = nn.Conv2d(planes, planes, 3, 1, 1, bias=True)
        self.conv2b.weight.data.fill_(0)
        self.conv2b.bias.data.fill_(0)

        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = LambdaLayer(
                lambda x: F.pad(x[:, :, ::2, ::2],
                                (0, 0, 0, 0, planes//4, planes//4))
            )

    def forward(self, x):
        relu = self.config.relu_mode

        out = self.bn1(self.conv1r(x)*self.conv1g(x) + self.conv1b(x.pow(2)))

        if relu == 2:
            out = F.relu(out)

        out = self.bn2(self.conv2r(out)*self.conv2g(out) + self.conv2b(out.pow(2)))
        out += self.shortcut(x)

        if relu in [1, 2]:
            out = F.relu(out)

        return out


class QResNet(nn.Module):
    def __init__(self, config, block, num_blocks):
        super().__init__()
        self.config = config
        self.in_planes = 16

        self.conv1 = nn.Conv2d(config.input_channels, 16, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)

        self.layer1 = self._make_layer(block, 16, num_blocks[0])
        self.layer2 = self._make_layer(block, 32, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 64, num_blocks[2], stride=2)

        self.linear = nn.Linear(64, config.num_classes)

    def _make_layer(self, block, planes, num_blocks, stride=1):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []

        for s in strides:
            layers.append(block(self.in_planes, planes, self.config, s))
            self.in_planes = planes

        return nn.Sequential(*layers)

    def forward(self, x):
        if self.config.relu_mode == 0:
            out = self.bn1(self.conv1(x))
        else:
            out = F.relu(self.bn1(self.conv1(x)))

        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)

        out = F.avg_pool2d(out, out.size(3))
        out = out.view(out.size(0), -1)

        return self.linear(out)


def load_pretrained(model, pretrained_path):

    """ Load pretrained weights and freeze linear (r/g) convolutions
    
    Args:
        model: Your QResNet model instance
        pretrained_path: Path to pretrained .pth file
        freeze_linear: If True, freeze r and g conv layers
    """
    # Load pretrained weights
    pretrained_dict = torch.load(pretrained_path, map_location='cpu')
    model.load_state_dict(pretrained_dict)

    for name, param in model.named_parameters():
        if ('conv1r' in name or 'conv1g' in name or 
            'conv2r' in name or 'conv2g' in name):
            param.requires_grad = False
    
    return model


def qresnet20(config):
    return QResNet(config, BasicBlock, [3, 3, 3])

In [77]:
def train_model(X, y, config=config, model_fn=None):
    loader = DataLoader(
        TensorDataset(X, y),
        batch_size=config.batch_size,
        shuffle=True
    )
    if config.load_pretrained:
        model = load_pretrained(model_fn(config), pretrained_path=config.pretrained_path)
    else:
        model = model_fn(config)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Only optimize parameters that require gradients
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=config.lr)
    criterion = nn.CrossEntropyLoss()

    # Verify what's being trained
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable: {trainable_params:,} / {total_params:,} params ({100*trainable_params/total_params:.1f}%)")

    for epoch in range(config.epochs):
        model.train()
        total, correct, loss_sum = 0, 0, 0

        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item() * labels.size(0)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        print(f"Epoch {epoch+1}/{config.epochs} | "
              f"Loss: {loss_sum/total:.4f} | Acc: {correct/total:.4f}")

    return model

In [78]:
if config.use_mnist:
    df = pd.read_csv("data/mnist_train.csv", header=None)

    y = torch.tensor(df.iloc[:, 0].values, dtype=torch.long)
    X = torch.tensor(df.iloc[:, 1:].values, dtype=torch.float32) / 255.0  # normalize to [0, 1]

    num_pixels = X.shape[1]
    side = int(num_pixels ** 0.5)
    assert side * side == num_pixels, f"Expected square image, got {num_pixels} pixels."
    X = X.view(-1, 1, side, side)  # NCHW for Conv2d: [batch, channel, height, width]

In [79]:
if config.use_cifar10:
    X, y = load_batch("data/cifar-10-batches-py/data_batch_2")
    X = torch.from_numpy(X).float() / 255.0  # Normalize to [0, 1]
    y = torch.from_numpy(y).long()

In [80]:
def test_model(path, model):
    model.eval()  # disable dropout/batchnorm training behavior
    correct = 0
    total = 0

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    X, y = load_batch(path)
    X = torch.from_numpy(X).float() / 255.0  # Normalize to [0, 1]
    y = torch.from_numpy(y).long()

    train_loader = DataLoader(TensorDataset(X, y), batch_size=128, shuffle=True)

    with torch.no_grad():  # disable gradient computation
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)  # get class with highest score
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Accuracy: {accuracy:.2f}%')
    model.train()

In [81]:
qresnet20_model = train_model(X=X, y=y, model_fn=qresnet20)

C:\Users\Erik Li\AppData\Local\Temp\ipykernel_26720\3294534937.py:118: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained_dict = torch.load(pretrained_path, map_locati

Trainable: 270,394 / 806,266 params (33.5%)
Epoch 1/50 | Loss: 0.7968 | Acc: 0.7440
Epoch 2/50 | Loss: 0.4446 | Acc: 0.8453
Epoch 3/50 | Loss: 0.3454 | Acc: 0.8774
Epoch 4/50 | Loss: 0.2747 | Acc: 0.9034
Epoch 5/50 | Loss: 0.1969 | Acc: 0.9301
Epoch 6/50 | Loss: 0.1591 | Acc: 0.9433
Epoch 7/50 | Loss: 0.1284 | Acc: 0.9544
Epoch 8/50 | Loss: 0.1064 | Acc: 0.9624
Epoch 9/50 | Loss: 0.1023 | Acc: 0.9637
Epoch 10/50 | Loss: 0.0856 | Acc: 0.9698
Epoch 11/50 | Loss: 0.0860 | Acc: 0.9691
Epoch 12/50 | Loss: 0.0809 | Acc: 0.9713
Epoch 13/50 | Loss: 0.1966 | Acc: 0.9310
Epoch 14/50 | Loss: 0.0805 | Acc: 0.9724
Epoch 15/50 | Loss: 0.0511 | Acc: 0.9830
Epoch 16/50 | Loss: 0.3135 | Acc: 0.8943
Epoch 17/50 | Loss: 0.1330 | Acc: 0.9528
Epoch 18/50 | Loss: 0.0536 | Acc: 0.9829
Epoch 19/50 | Loss: 0.0454 | Acc: 0.9849
Epoch 20/50 | Loss: 0.0453 | Acc: 0.9846
Epoch 21/50 | Loss: 0.0652 | Acc: 0.9771
Epoch 22/50 | Loss: 0.0594 | Acc: 0.9793
Epoch 23/50 | Loss: 0.0472 | Acc: 0.9840
Epoch 24/50 | Loss: 0.

In [82]:
# qresnet20_model = load_model("qresnet20_noReLU.pth", qresnet20)

In [83]:
# save_model(qresnet20_model, "qresnet20")

In [84]:
test_model("data/cifar-10-batches-py/test_batch", qresnet20_model)

Accuracy: 69.67%


In [ ]:
# import json
# from pathlib import Path

# def notebook_to_clean_py(notebook_path, output_path=None):
#     notebook_path = Path(notebook_path)
#     if output_path is None:
#         output_path = notebook_path.with_suffix(".py")
#     else:
#         output_path = Path(output_path)

#     with notebook_path.open("r", encoding="utf-8") as f:
#         nb = json.load(f)

#     code_chunks = []
#     for cell in nb.get("cells", []):
#         if cell.get("cell_type") == "code":
#             source = cell.get("source", [])
#             if isinstance(source, list):
#                 code = "".join(source)
#             else:
#                 code = source
#             if code.strip():
#                 code_chunks.append(code.rstrip())

#     output_text = "\n\n".join(code_chunks) + "\n"

#     with output_path.open("w", encoding="utf-8") as f:
#         f.write(output_text)

#     print(f"Saved clean Python file to: {output_path}")

# # Example usage:
# notebook_to_clean_py("Quad_Neuron.ipynb")
# # notebook_to_clean_py("input.ipynb", "output.py")

Saved clean Python file to: Quad_Neuron.py
